# Círculos de Mohr 3D — Visualización interactiva

En el caso general, el estado tensional en un punto se describe con un **tensor de tensiones simétrico** de 3×3:

$$\boldsymbol{\sigma} = \begin{pmatrix} \sigma_x & \tau_{xy} & \tau_{xz} \\ \tau_{xy} & \sigma_y & \tau_{yz} \\ \tau_{xz} & \tau_{yz} & \sigma_z \end{pmatrix}$$

es decir, **6 componentes independientes**.

## ¿Qué pasa al girar el elemento en 3D?

Igual que en 2D: el estado tensional **es el mismo**, solo cambia la orientación de los planos sobre los que medimos las tensiones. La rotación se describe ahora con tres ángulos $(\alpha, \beta, \gamma)$ alrededor de los ejes $x$, $y$, $z$ globales.

Si $R(\alpha, \beta, \gamma)$ es la matriz de rotación, el tensor de tensiones en el sistema rotado se obtiene como:

$$\boldsymbol{\sigma}' = R^{T}\,\boldsymbol{\sigma}\,R$$

Las **caras del cubo girado** tienen normales que son las columnas de $R$. Sobre cada cara aparecen una tensión normal $\sigma_n$ y un vector cortante $\vec{\tau}$ contenido en el plano de la cara.

## Los tres círculos de Mohr

Diagonalizando el tensor obtenemos las **tensiones principales** $\sigma_1 \ge \sigma_2 \ge \sigma_3$ (autovalores) y sus direcciones (autovectores).

En el plano $(\sigma, \tau)$ se dibujan tres semicírculos:

- $C_{12}$: centro $\tfrac{\sigma_1+\sigma_2}{2}$, radio $\tfrac{\sigma_1-\sigma_2}{2}$
- $C_{23}$: centro $\tfrac{\sigma_2+\sigma_3}{2}$, radio $\tfrac{\sigma_2-\sigma_3}{2}$
- $C_{13}$: centro $\tfrac{\sigma_1+\sigma_3}{2}$, radio $\tfrac{\sigma_1-\sigma_3}{2}$ (el grande)

**Para cualquier orientación posible del plano de corte**, el par $(\sigma_n, |\tau|)$ cae **dentro de la región amarilla**: limitada por fuera por $C_{13}$ y por dentro por $C_{12}$ y $C_{23}$.

La **cortante máxima** absoluta vale $\tau_{\max} = \tfrac{\sigma_1 - \sigma_3}{2}$ (radio del círculo grande).

## Caso 2D como caso particular

El estado plano corresponde a poner $\sigma_z = \tau_{xz} = \tau_{yz} = 0$ y rotar solo $\gamma$ (alrededor de $z$). Verás cómo los puntos de las caras x' e y' recorren uno de los tres círculos, mientras la cara z' se queda fija en $\sigma_3 = 0$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registra la proyección 3D)
from ipywidgets import interact, FloatSlider, Layout

%matplotlib inline

## Funciones auxiliares

- `matriz_tension`: construye el tensor de tensiones simétrico a partir de las 6 componentes.
- `matriz_rotacion`: matriz de rotación 3D a partir de los ángulos $(\alpha, \beta, \gamma)$ alrededor de $x$, $y$, $z$ (extrínsecos, en orden $R_z R_y R_x$).
- `transformar_tensor`: aplica $\boldsymbol{\sigma}' = R^{T}\boldsymbol{\sigma}R$.
- `tensiones_principales_3d`: devuelve $\sigma_1 \ge \sigma_2 \ge \sigma_3$ y sus direcciones.
- `estado_caras`: para cada cara del cubo girado calcula $(\sigma_n, |\tau_n|)$, los puntos que se dibujan sobre el diagrama de Mohr.

In [2]:
def matriz_tension(sx, sy, sz, txy, txz, tyz):
    """Tensor de tensiones simétrico 3x3."""
    return np.array([[sx,  txy, txz],
                     [txy, sy,  tyz],
                     [txz, tyz, sz ]], dtype=float)

def matriz_rotacion(alpha_deg, beta_deg, gamma_deg):
    """Rotación R = Rz(γ) · Ry(β) · Rx(α). Las columnas de R son los ejes del
    sistema rotado expresados en el sistema global."""
    a, b, g = np.deg2rad([alpha_deg, beta_deg, gamma_deg])
    Rx = np.array([[1, 0,        0       ],
                   [0, np.cos(a),-np.sin(a)],
                   [0, np.sin(a), np.cos(a)]])
    Ry = np.array([[ np.cos(b), 0, np.sin(b)],
                   [ 0,         1, 0        ],
                   [-np.sin(b), 0, np.cos(b)]])
    Rz = np.array([[np.cos(g),-np.sin(g), 0],
                   [np.sin(g), np.cos(g), 0],
                   [0,         0,         1]])
    return Rz @ Ry @ Rx

def transformar_tensor(T, R):
    """σ' = Rᵀ σ R."""
    return R.T @ T @ R

def tensiones_principales_3d(T):
    """Devuelve (σ1, σ2, σ3) ordenadas de mayor a menor y sus direcciones."""
    vals, vecs = np.linalg.eigh(T)   # eigh: simétrico → autovalores reales
    idx = np.argsort(vals)[::-1]
    return vals[idx], vecs[:, idx]

def estado_caras(Tp):
    """Para cada cara del cubo girado (normales e1', e2', e3' del sistema rotado),
    devuelve (σ_n, |τ_n|): la tensión normal y la magnitud de la cortante.

    El traction sobre la cara con normal e_i' es t' = Tp · e_i' = columna i de Tp.
    σ_n = t'·e_i' = Tp[i,i],   |τ_n| = √(|t'|² − σ_n²)
    """
    out = []
    for i in range(3):
        t = Tp[:, i]
        sn = Tp[i, i]
        tn = np.sqrt(max(np.dot(t, t) - sn**2, 0.0))
        out.append((sn, tn))
    return out

In [3]:
def dibujar_cubo_3d(ax, T0, R):
    """Dibuja el cubo original (gris) y el cubo girado, con flechas de tensión
    normal (rojo) y cortante (azul) sobre las caras +x', +y', +z'."""
    ax.clear()
    L = 1.0
    Tp = transformar_tensor(T0, R)

    # --- Vértices del cubo unitario y aristas ---
    corners0 = np.array([[a, b, c]
                         for a in (-L, L) for b in (-L, L) for c in (-L, L)])
    edges_idx = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),
                 (2,6),(3,7),(4,5),(4,6),(5,7),(6,7)]

    # Cubo de referencia (sin girar)
    for i, j in edges_idx:
        ax.plot(*zip(corners0[i], corners0[j]),
                color='gray', lw=0.6, ls=':', alpha=0.4)

    # Cubo girado: aplicamos R a cada vértice (vector global = R · vector local)
    corners = corners0 @ R.T
    for i, j in edges_idx:
        ax.plot(*zip(corners[i], corners[j]),
                color='black', lw=1.6)

    # --- Tensiones en cada cara del cubo girado ---
    smax = max(np.abs(T0).max(), 1e-9)
    k = 0.7 / smax  # escala visual de las flechas

    colores_caras = ['#d62728', '#2ca02c', '#1f77b4']  # rojo, verde, azul
    nombres_caras = ["x'", "y'", "z'"]

    for i in range(3):
        # Eje del sistema rotado (en coordenadas globales)
        n  = R[:, i]
        # Vectores tangentes (los otros dos ejes rotados)
        ej = R[:, (i+1) % 3]
        ek = R[:, (i+2) % 3]

        center_pos =  L * n
        center_neg = -L * n

        sigma_n = Tp[i, i]
        # Vector cortante en sistema global = suma de las componentes tangenciales
        shear_vec = Tp[(i+1) % 3, i] * ej + Tp[(i+2) % 3, i] * ek

        # Flecha de tensión normal (rojo): cara positiva sale, negativa entra (signo gestiona tracción/compresión)
        ax.quiver(*center_pos, *(sigma_n*k*n),
                  color='red', lw=2.0, arrow_length_ratio=0.18, normalize=False)
        ax.quiver(*center_neg, *(-sigma_n*k*n),
                  color='red', lw=2.0, arrow_length_ratio=0.18, normalize=False)

        # Flecha de tensión cortante (azul) — solo si tiene magnitud apreciable
        if np.linalg.norm(shear_vec) > 1e-6 * smax:
            ax.quiver(*center_pos, *(shear_vec*k),
                      color='blue', lw=2.0, arrow_length_ratio=0.18, normalize=False)
            ax.quiver(*center_neg, *(-shear_vec*k),
                      color='blue', lw=2.0, arrow_length_ratio=0.18, normalize=False)

        # Etiqueta del eje rotado
        ax.text(*(1.55*n), nombres_caras[i], color=colores_caras[i],
                fontsize=11, fontweight='bold')

    # --- Ejes globales (gris) ---
    for i, label in enumerate(['x', 'y', 'z']):
        e = np.zeros(3); e[i] = 2.3
        ax.quiver(0, 0, 0, *e, color='gray', lw=1, arrow_length_ratio=0.06)
        ax.text(*(e*1.08), label, color='gray', fontsize=10)

    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_zlim(-2.5, 2.5)
    try:
        ax.set_box_aspect([1, 1, 1])  # matplotlib >= 3.3
    except AttributeError:
        pass
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_title('Cubo girado y tensiones en sus caras')

In [4]:
def dibujar_mohr_3d(ax, T0, R):
    """Dibuja los tres semicírculos de Mohr, sombrea la región admisible y
    marca los puntos (σ_n, |τ_n|) correspondientes a las tres caras del cubo girado."""
    ax.clear()
    Tp = transformar_tensor(T0, R)
    s_arr, _ = tensiones_principales_3d(T0)
    s1, s2, s3 = s_arr  # s1 ≥ s2 ≥ s3

    # Centros y radios
    C12, R12 = (s1 + s2) / 2, (s1 - s2) / 2
    C23, R23 = (s2 + s3) / 2, (s2 - s3) / 2
    C13, R13 = (s1 + s3) / 2, (s1 - s3) / 2

    # --- Sombrear región admisible ---
    if R13 > 1e-9:
        margen = 0.15 * R13 + 0.5
        xs = np.linspace(s3 - margen, s1 + margen, 500)
        ys = np.linspace(0,             R13 + margen, 300)
        X, Y = np.meshgrid(xs, ys)
        adentro_grande = (X - C13)**2 + Y**2 <= R13**2
        afuera_12      = (X - C12)**2 + Y**2 >= R12**2
        afuera_23      = (X - C23)**2 + Y**2 >= R23**2
        region = adentro_grande & afuera_12 & afuera_23
        ax.contourf(X, Y, region.astype(float),
                    levels=[0.5, 1.5], colors=['#fff3b0'], alpha=0.55)

    # --- Tres semicírculos ---
    th = np.linspace(0, np.pi, 200)
    ax.plot(C12 + R12*np.cos(th), R12*np.sin(th),
            color='#1f77b4', lw=1.6, label=f'C₁₂  R={R12:.1f}')
    ax.plot(C23 + R23*np.cos(th), R23*np.sin(th),
            color='#2ca02c', lw=1.6, label=f'C₂₃  R={R23:.1f}')
    ax.plot(C13 + R13*np.cos(th), R13*np.sin(th),
            color='#d62728', lw=1.8, label=f'C₁₃  R={R13:.1f}  (máx)')

    # --- Tensiones principales ---
    for sigma, lab in zip([s1, s2, s3],
                          [r'$\sigma_1$', r'$\sigma_2$', r'$\sigma_3$']):
        ax.plot(sigma, 0, 'ko', markersize=6, zorder=4)
        ax.annotate(f'{lab}={sigma:.1f}', xy=(sigma, 0),
                    xytext=(0, -14), textcoords='offset points',
                    ha='center', va='top', fontsize=9)

    # --- Puntos del estado actual (caras del cubo girado) ---
    estado = estado_caras(Tp)
    colores = ['#d62728', '#2ca02c', '#1f77b4']
    nombres = ["x'", "y'", "z'"]
    for i, (sn, tn) in enumerate(estado):
        ax.plot(sn, tn, 'o', color=colores[i], markersize=11, zorder=6,
                markeredgecolor='black', markeredgewidth=0.6)
        ax.annotate(f"  {nombres[i]} ({sn:.1f}, {tn:.1f})",
                    xy=(sn, tn), color=colores[i], fontsize=9, va='center')

    ax.axhline(0, color='gray', lw=0.7)
    ax.set_aspect('equal')
    ax.set_xlabel(r'$\sigma$  (tensión normal sobre el plano)')
    ax.set_ylabel(r'$|\tau|$  (cortante sobre el plano)')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper right', fontsize=8)
    ax.set_title(rf'Círculos de Mohr 3D · $\tau_{{máx}} = {R13:.2f}$')

## Visualización interactiva

Mueve los **9 deslizadores**:

- **6 componentes del tensor de tensiones inicial**: $\sigma_x, \sigma_y, \sigma_z$ (normales) y $\tau_{xy}, \tau_{xz}, \tau_{yz}$ (cortantes).
- **3 ángulos de rotación** del cubo: $\alpha$ (alrededor de $x$), $\beta$ (alrededor de $y$), $\gamma$ (alrededor de $z$).

En la **leyenda inferior** aparecen las 6 componentes del tensor en el sistema rotado $\boldsymbol{\sigma}'$ — son las que actúan sobre las caras del cubo girado para los ángulos seleccionados.

En el **diagrama de Mohr 3D**:

- Los **tres semicírculos** dependen únicamente del estado tensional (no de la rotación).
- Los **tres puntos de colores** corresponden a las tres caras del cubo girado:
  - Punto **rojo** → cara x' del cubo girado: $(\sigma_{x'}, |\tau|_{cara\;x'})$
  - Punto **verde** → cara y'
  - Punto **azul** → cara z'
- Los puntos se mueven dentro de la región amarilla al cambiar los ángulos. Los círculos no cambian.
- Cuando los tres puntos caen sobre el eje $\sigma$ (con $|\tau|=0$), el cubo está alineado con las **direcciones principales**: las caras solo soportan tensión normal.

In [5]:
def actualizar(sigma_x=80, sigma_y=20, sigma_z=0,
               tau_xy=30, tau_xz=0, tau_yz=0,
               alpha=0, beta=0, gamma=0):
    T0 = matriz_tension(sigma_x, sigma_y, sigma_z, tau_xy, tau_xz, tau_yz)
    R  = matriz_rotacion(alpha, beta, gamma)
    Tp = transformar_tensor(T0, R)
    s_arr, _ = tensiones_principales_3d(T0)

    fig = plt.figure(figsize=(15, 7))
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax2 = fig.add_subplot(1, 2, 2)
    dibujar_cubo_3d(ax1, T0, R)
    dibujar_mohr_3d(ax2, T0, R)

    # --- Leyenda con las 6 componentes del tensor en el sistema rotado ---
    leyenda = (
        "TENSOR INICIAL  σ              TENSOR ROTADO  σ'\n"
        f" σ_x  = {sigma_x:+7.2f}            σ_x' = {Tp[0,0]:+7.2f}\n"
        f" σ_y  = {sigma_y:+7.2f}            σ_y' = {Tp[1,1]:+7.2f}\n"
        f" σ_z  = {sigma_z:+7.2f}            σ_z' = {Tp[2,2]:+7.2f}\n"
        f" τ_xy = {tau_xy:+7.2f}            τ_x'y' = {Tp[0,1]:+7.2f}\n"
        f" τ_xz = {tau_xz:+7.2f}            τ_x'z' = {Tp[0,2]:+7.2f}\n"
        f" τ_yz = {tau_yz:+7.2f}            τ_y'z' = {Tp[1,2]:+7.2f}\n"
        f"\n Ángulos: α = {alpha:+.0f}°, β = {beta:+.0f}°, γ = {gamma:+.0f}°"
        f"        Principales:  σ₁={s_arr[0]:+.2f}  σ₂={s_arr[1]:+.2f}  σ₃={s_arr[2]:+.2f}"
    )
    fig.text(0.01, -0.02, leyenda, fontsize=10, family='monospace',
             va='top', ha='left',
             bbox=dict(boxstyle='round,pad=0.5',
                       facecolor='white', edgecolor='gray', alpha=0.95))

    plt.tight_layout()
    plt.show()


estilo = {'description_width': '70px'}
ancho  = Layout(width='340px')

interact(
    actualizar,
    sigma_x=FloatSlider(min=-100, max=100, step=5, value=80,
                        description='σx',  style=estilo, layout=ancho),
    sigma_y=FloatSlider(min=-100, max=100, step=5, value=20,
                        description='σy',  style=estilo, layout=ancho),
    sigma_z=FloatSlider(min=-100, max=100, step=5, value=0,
                        description='σz',  style=estilo, layout=ancho),
    tau_xy=FloatSlider(min=-80,  max=80,  step=5, value=30,
                       description='τxy', style=estilo, layout=ancho),
    tau_xz=FloatSlider(min=-80,  max=80,  step=5, value=0,
                       description='τxz', style=estilo, layout=ancho),
    tau_yz=FloatSlider(min=-80,  max=80,  step=5, value=0,
                       description='τyz', style=estilo, layout=ancho),
    alpha=FloatSlider(min=-90,  max=90,  step=1, value=0,
                      description='α (x)°', style=estilo, layout=ancho),
    beta =FloatSlider(min=-90,  max=90,  step=1, value=0,
                      description='β (y)°', style=estilo, layout=ancho),
    gamma=FloatSlider(min=-90,  max=90,  step=1, value=0,
                      description='γ (z)°', style=estilo, layout=ancho),
);

interactive(children=(FloatSlider(value=80.0, description='σx', layout=Layout(width='340px'), min=-100.0, step…

## Casos para probar

1. **Caso 2D recuperado (estado plano)**: pon $\sigma_z = \tau_{xz} = \tau_{yz} = 0$ y solo mueve $\gamma$. Verás que el punto z' se queda fijo en el origen ($\sigma_z = 0$, sin cortante) y los puntos x' e y' recorren uno de los tres círculos. Los otros dos círculos pasan por el origen.

2. **Tracción uniaxial pura**: $\sigma_x = 100$, todas las demás 0. Las tensiones principales son $(100, 0, 0)$. Solo aparece **un círculo** (los otros dos coinciden). $\tau_{\max} = 50$ se alcanza girando 45°.

3. **Estado hidrostático**: $\sigma_x = \sigma_y = \sigma_z = 50$, todas las cortantes 0. **Los tres círculos degeneran en un único punto**: gires lo que gires, los tres puntos del cubo se quedan en $(\sigma=50, \tau=0)$. Por eso una presión hidrostática nunca produce deslizamiento.

4. **Cortante 3D genérico**: $\sigma_x = 80, \sigma_y = 20, \sigma_z = -30, \tau_{xy} = 25, \tau_{xz} = 15, \tau_{yz} = 10$. Mueve los tres ángulos: observa que los puntos siempre se quedan en la región amarilla (es físicamente imposible salir de ella).

5. **Encontrar las direcciones principales**: con cualquier estado de tensión, ajusta $(\alpha, \beta, \gamma)$ hasta que los tres puntos caigan sobre el eje $\sigma$. Los valores $\sigma_{x'}, \sigma_{y'}, \sigma_{z'}$ que aparecen en la leyenda coinciden entonces con $\sigma_1, \sigma_2, \sigma_3$ y todas las cortantes son cero.

6. **Cortante puro 3D**: $\sigma_x = \sigma_y = \sigma_z = 0$, $\tau_{xy} = \tau_{xz} = \tau_{yz} = 50$. Las principales salen $\sigma_1 = 100, \sigma_2 = \sigma_3 = -50$ — un círculo grande y dos coincidentes.